# MUSANGA V3.1 · NB06 — PlanetScope & embedding spatial information

**Question:** Does fine-scale spatial context add meaningful and transferable information for estimating *Musanga* fraction beyond the established Sentinel-2 and satellite-embedding baseline?

**Stage 1 — exploration, not model fitting:** establish native resolution, available 4-band Planet annual/monthly imagery, annual 9-band NIR texture, 64-band 10 m embeddings and the existing 100 m UAV target cells. Display matched Planet and embedding patches, inspect 30/50/100 m supervision availability and spatial alignment. Compare native PlanetScope canopy texture with spatial variability in embeddings without confusing apparent training fit with generalization.

**Stage 2 — planned ablation, only after patch QA:** model comparison under identical targets and folds:
- **B0:** S2 annual + S2 quarterly + annual satellite embeddings (NB04B baseline).
- **B1:** B0 + handcrafted Planet texture (existing NB05 features).
- **B2:** B0 + multiscale **spatial texture of embeddings** (not merely their 64 central-pixel values).
- **B3:** B0 + Planet spatial CNN features.
- **B4:** B0 + embedding spatial CNN features.
- **B5:** B0 + both Planet and embedding CNN features.

First pilot at 50 m, then 30/100 m. **Same continuous UAV target**, same spatially held-out folds for within-site testing and leave-one-forest-out for geographic transfer. Hold out complete spatial areas/sites *before* patch extraction or learned representation training. Run supervised CNN embedding extraction out-of-fold, and fit all scalers/PCA/normalization from training folds only.

**Constraints:** Annual Planet image is 4.77 m and **may not be cloud-free**. Satellite embeddings are 10 m and already summarize S1/S2 information. Yoko Planet 2021 vs UAV 2020 is a known mismatch. Existing NB04B 100 m samples are used for preview only; no CNN trained in this first stage. No full-ROI 4.77 m imagery is assumed downloaded.


## 00 · Settings — exploratory preview only


In [ ]:
from pathlib import Path
FINAL=next((p for p in [Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
                          Path('/content/drive/MyDrive/Musanga/Final')] if p.is_dir()),None)
if FINAL is None:
    from google.colab import drive
    drive.mount('/content/drive')
    FINAL=next((p for p in [Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
                              Path('/content/drive/MyDrive/Musanga/Final')] if p.is_dir()),None)
assert FINAL and FINAL.is_dir(),'Musanga Final workspace missing'
ROOT=FINAL/'Outputs/V3_1'
NB05=ROOT/'05_sat_predictor_exploration'
NB04=ROOT/'04_satellite_predictors/input_layers'
NB04B=ROOT/'04B_roi_quarterly_LOSO'
OUT=ROOT/'06_spatial_planet_embeddings'
FIG=OUT/'figures'
for d in (OUT,FIG):d.mkdir(parents=True,exist_ok=True)
SITES=['Baego','MOS','Yangambi','Yoko']
PLANET_YEAR={'Baego':2023,'MOS':2024,'Yangambi':2024,'Yoko':2021}
EMBEDDING_YEAR={'Baego':2023,'MOS':2024,'Yangambi':2024,'Yoko':2020}
PREVIEW_PER_SITE=3
PREVIEW_PATCH_PIXELS=48      # ~229m Planet context for each selected 100m cell
EMB_PATCH_PIXELS=23          # ~230m at 10m embedding resolution
RANDOM_SEED=20261024
print('Workspace:',FINAL)


## 01 · Validate source products and review temporal/resolution limitations

Checks actual files, expected 19-band annual Planet, 9-band NIR texture and the existing 64-band annual embeddings. Annual Planet composition bands are:
1–4 screened mean B/G/R/N; 5–8 screened median; 9–12 raw mean; **13–16 raw median**; 17–19 raw valid count / screened valid count / number screened out.
The screened product is **heuristic only**; use the raw median as the stable first CNN baseline.

NB04B's original 100 m sample cache is *not recreated* or overwritten.


In [ ]:
import numpy as np,pandas as pd,rasterio,matplotlib.pyplot as plt
from rasterio.windows import Window
from rasterio.warp import transform as rio_transform
from rasterio.enums import Resampling
from pyproj import Transformer
from IPython.display import display

def planet_file(site,kind):
    root=NB05/('derived_features' if kind=='Texture' else 'input_layers')
    folder=f'Musanga_V31_NB05_UAV_ROI_{kind}'
    tag=f'NB05_UAV_ROI_{site}_{PLANET_YEAR[site]}_'
    pattern=tag+('Annual_19band*.tif' if kind=='Annual' else 'Texture_9band*.tif')
    matches=sorted((root/folder).glob(pattern))
    if len(matches)!=1:
        raise FileNotFoundError(f'{site} {kind}: expected one file, found {len(matches)}: {matches}')
    return matches[0]

def embed_file(site):
    return NB04/'EMB_ANNUAL'/f'{site}_EMB_ANNUAL_{EMBEDDING_YEAR[site]}_10m.tif'

AUDIT=[]
for site in SITES:
    for family,filepath,expected in (
        ('PLANET_ANNUAL',planet_file(site,'Annual'),19),
        ('PLANET_TEXTURE',planet_file(site,'Texture'),9),
        ('EMB_ANNUAL',embed_file(site),64)):
        assert filepath.is_file(),filepath
        with rasterio.open(filepath) as src:
            assert src.count==expected,(site,family,'expected',expected,'found',src.count)
            assert src.crs and src.width>0 and src.height>0
            AUDIT.append(dict(site=site,family=family,year=PLANET_YEAR[site] if family!='EMB_ANNUAL' else EMBEDDING_YEAR[site],
                bands=src.count,pixel_size_m=round(abs(src.res[0]),3),
                width=src.width,height=src.height,crs=str(src.crs),
                size_MB=round(filepath.stat().st_size/1e6,2),path=str(filepath)))
AUDIT=pd.DataFrame(AUDIT)
display(AUDIT.drop(columns=['path']))
AUDIT.to_csv(OUT/'01_input_audit.csv',index=False)
assert (NB05/'07_final_input_audit/07_site_summary.csv').exists(),'Complete NB05 Section 07 audit first'
print('All 12 native products found: four forests × annual Planet/texture/embeddings.')
print('Note: The 10m embeddings are not 4.77m imagery; do not upscale them to claim new spatial detail.')


## 02 · Select existing 100 m UAV reference cells (identical NB04B targets)

Only select example cells; avoid random point selection from the full Planet image, which would include areas without UAV supervision. Sampling is stratified by low/medium/high Musanga abundance **within each site**, so each forest's previews show more than zero-cover background. Cell centers are computed from NB04B's exact S2-grid transform, not from row/col as if they were Planet raster indices.


In [ ]:
from affine import Affine
SAMPLE_PATH=NB04B/'samples/roi_cells_100m.parquet'
assert SAMPLE_PATH.is_file(),SAMPLE_PATH
DF=pd.read_parquet(SAMPLE_PATH)
assert {'site','row','col','mean_probability'}.issubset(DF.columns)
DF=DF.loc[DF.site.isin(SITES)].copy()
assert len(DF)>0
RNG=np.random.default_rng(RANDOM_SEED)
CENTERS={}
SELECTED=[]
for site in SITES:
    local=DF.loc[DF.site==site].copy()
    assert not local.empty,site
    # Existing 100m NB04B samples use the 10m S2 affine aggregated exactly 10×.
    raw=NB04/'RAW_ANNUAL'/f'{site}_RAW_ANNUAL_{EMBEDDING_YEAR[site]}_10m.tif'
    assert raw.is_file(),raw
    with rasterio.open(raw) as ds:
        assert abs(ds.res[0]-10)<.05 and abs(ds.res[1]-10)<.05
        tfm=ds.transform*Affine.scale(10,10)
        crs=ds.crs
    for i in range(PREVIEW_PER_SITE):
        ordered=local.sort_values('mean_probability')
        idx=int(round((i+.5)*(len(ordered)-1)/PREVIEW_PER_SITE))
        sample=ordered.iloc[idx]
        cx,cy=tfm*(float(sample.col)+.5,float(sample.row)+.5)
        SELECTED.append(dict(site=site,group=str(sample.group),mosaic=str(sample.mosaic),
            fraction=float(sample.mean_probability),row=int(sample.row),col=int(sample.col),
            center_x=float(cx),center_y=float(cy),crs=str(crs)))
    CENTERS[site]=dict(transform=tfm,crs=crs)
PREVIEW=pd.DataFrame(SELECTED)
display(PREVIEW[['site','group','fraction','row','col']].round(3))
PREVIEW.to_csv(OUT/'02_preview_cells.csv',index=False)
print('Preview targets are UAV-model estimates at 100 m, NOT independent field labels.')


## 03 · Matched *native* Planet RGB/NIR and embedding-neighborhood previews

For the same UAV-target cell center, read a ~229 m **Planet raw annual median RGB patch** and a ~230 m **10 m satellite-embedding patch**. Embeddings contain 64 latent bands; visualize the first three PCA components as an exploratory false-colour composite (PCA fit only on the displayed patch, **not** for any prediction model). Preserve image pixel scales. This step helps spot spatial shifts, boundary masking, and structural patterns.

The common center is transformed to each raster's CRS if necessary. If a patch crosses a source footprint boundary it is padded with zeros and flagged by nonzero-band masks; no fabricated reflectance values.


In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
def native_patch(path,cx,cy,center_crs,size,bands):
    with rasterio.open(path) as ds:
        trans=Transformer.from_crs(center_crs,ds.crs,always_xy=True)
        x,y=trans.transform(cx,cy)
        cc,rr=~ds.transform*(x,y)
        start_col=int(np.floor(cc-size/2))
        start_row=int(np.floor(rr-size/2))
        w=Window(start_col,start_row,size,size)
        arr=ds.read(bands,window=w,boundless=True,fill_value=0).astype('float32')
        return arr,ds.crs,ds.res
def rgb_fixed(arr,valid):
    # Input arr is RGB band order; calculate one robust stretch per local patch.
    if np.count_nonzero(valid)<10:return np.ones((*valid.shape,3),dtype='float32')
    p=arr[:,valid]
    lo=np.percentile(p,2,axis=1)
    hi=np.percentile(p,98,axis=1)
    rgb=np.moveaxis(np.clip((arr-lo[:,None,None])/(hi[:,None,None]-lo[:,None,None]+1e-6),0,1),0,-1)
    rgb[~valid]=1
    return rgb
def embed_pca_rgb(cube):
    mask=np.isfinite(cube).all(axis=0)&np.any(cube!=0,axis=0)
    out=np.ones((*mask.shape,3),dtype='float32')
    if mask.sum()<10:return out,mask
    X=cube[:,mask].T
    X=StandardScaler().fit_transform(X)
    vals=PCA(n_components=3).fit_transform(X)
    pct_lo=np.percentile(vals,2,axis=0);pct_hi=np.percentile(vals,98,axis=0)
    out[mask]=np.clip((vals-pct_lo)/(pct_hi-pct_lo+1e-6),0,1)
    return out,mask

fig,axes=plt.subplots(len(PREVIEW),2,figsize=(8,3.0*len(PREVIEW)),layout='constrained')
rows=[]
for i,row in enumerate(PREVIEW.itertuples(index=False)):
    pa=planet_file(row.site,'Annual')
    eb=embed_file(row.site)
    # Planet annual 13-16 = B,G,R,N raw median; select RGB order 15,14,13
    p,_,pres=native_patch(pa,row.center_x,row.center_y,row.crs,PREVIEW_PATCH_PIXELS,[15,14,13])
    e,_,eres=native_patch(eb,row.center_x,row.center_y,row.crs,EMB_PATCH_PIXELS,list(range(1,65)))
    pmask=np.all(np.isfinite(p)&(p>0),axis=0)
    eviz,emask=embed_pca_rgb(e)
    axes[i,0].imshow(rgb_fixed(p,pmask));axes[i,1].imshow(eviz)
    axes[i,0].set_title(f'{row.site} | UAV {row.fraction*100:.1f}% | Planet RGB')
    axes[i,1].set_title('10 m embedding neighborhood (local PCA)')
    for ax in axes[i]:ax.axis('off')
    rows.append(dict(site=row.site,fraction=row.fraction,
                     planet_valid_pct=100*pmask.mean(),embedding_valid_pct=100*emask.mean(),
                     planet_pixel_size_m=abs(pres[0]),embedding_pixel_size_m=abs(eres[0])))
plt.savefig(FIG/'03_planet_vs_embedding_native_patches.png',dpi=140,bbox_inches='tight')
plt.show()
PATCH_QA=pd.DataFrame(rows)
display(PATCH_QA.round(2))
PATCH_QA.to_csv(OUT/'03_patch_coverage.csv',index=False)
print('IMPORTANT: local PCA colours are arbitrary across panels/sites and cannot be compared spectrally.')


## 04 · Spatial embeddings vs Planet: first quantitative exploratory features

To assess whether embeddings also contain spatial texture, summarize **within-patch heterogeneity** of their 64 channels. Compare each 100 m target cell's:
- Planet NIR local texture (from NB05's existing annual raster, sampled as local median);
- embedding spatial heterogeneity (median channelwise SD over a ~230 m context), plus embedding central-context mean differences.

**Descriptive only:** 12 deliberately stratified preview cells are too few for useful correlations or model fitting. This diagnostic checks that features are finite and nontrivial. Later we will extract these features across *all* labelled cells, use consistent physical windows (not just fixed pixel counts), and evaluate added value on geographic holdouts.


In [ ]:
from scipy.ndimage import median_filter
QA=[]
for row in PREVIEW.itertuples(index=False):
    e,_,_=native_patch(embed_file(row.site),row.center_x,row.center_y,row.crs,EMB_PATCH_PIXELS,list(range(1,65)))
    m=np.isfinite(e).all(axis=0)&np.any(e!=0,axis=0)
    if m.sum()<25:continue
    em=e[:,m]
    vari=np.std(em,axis=1)
    center=e[:,EMB_PATCH_PIXELS//2,EMB_PATCH_PIXELS//2]
    context=np.mean(em,axis=1)
    t,_,_=native_patch(planet_file(row.site,'Texture'),row.center_x,row.center_y,row.crs,
                       PREVIEW_PATCH_PIXELS,[1,2,3,4])
    tm=np.isfinite(t).all(axis=0)&(t>=0).all(axis=0)&np.any(t>0,axis=0)
    QA.append(dict(site=row.site,uav_fraction=row.fraction,
        embedding_channel_sd_median=float(np.median(vari)),
        embedding_context_difference=float(np.mean(abs(center-context))),
        planet_NIR_CV_r3_median=float(np.median(t[1,tm])) if tm.any() else np.nan,
        valid_embedding_pct=100*m.mean(),valid_planet_texture_pct=100*tm.mean()))
Q=pd.DataFrame(QA)
display(Q.round(4))
Q.to_csv(OUT/'04_spatial_feature_feasibility.csv',index=False)
fig,axes=plt.subplots(1,2,figsize=(10,4),layout='constrained')
for site,group in Q.groupby('site'):
    axes[0].scatter(group.uav_fraction*100,group.embedding_channel_sd_median,label=site)
    axes[1].scatter(group.uav_fraction*100,group.planet_NIR_CV_r3_median,label=site)
axes[0].set(xlabel='UAV Musanga fraction (%)',ylabel='Median channel SD in embedding patch',
            title='Embeddings: spatial heterogeneity')
axes[1].set(xlabel='UAV Musanga fraction (%)',ylabel='Planet NIR CV (radius 3)',
            title='Planet: native-scale texture')
axes[1].legend()
plt.savefig(FIG/'04_spatial_heterogeneity_feasibility.png',dpi=150,bbox_inches='tight')
plt.show()
print('This plot has only 12 stratified examples. Do NOT interpret it as prediction skill.')


## 05 · Decision gate and forthcoming modelling experiment

Before CNN model fitting, confirm:
1. Patch-validity percentages and visual alignment of Planet with its target cells; independent UAV orthomosaic/probability overlay will follow when locating the original teacher rasters.
2. Check that Planet **2021** in Yoko is an intentional temporal mismatch against UAV **2020**.
3. Use 50 m targets first and compare 30/100 m; target extraction must use the original **full-resolution UAV probability rasters**, avoiding interpolation of 100 m labels.
4. Train compact models with group/spatial folds: **100 m/50 m patches overlap heavily**, so randomly split cells is not an acceptable test.
5. Compare `B0 → B1/B2 → B3/B4 → B5` using the same site holdout and identical target cells; summarize MAE (pp), RMSE (pp), R², high-cover bias, calibration, and performance separately at MOS. Aggregate CNN output to target cells fairly.

**NB06 currently implements the exploratory entry point (Sections 00–04) and experimental specification; CNN training follows after visual QA.** Avoid building a large training architecture before verifying supervision and patch geometry.


## 06 · Verify original UAV probability rasters and Planet co-registration

The authoritative teacher rasters are the same **M2 warmstart full-mosaic U-Net probability GeoTIFFs** used by NB04 (not the 100 m aggregate). We inspect their CRS, footprint, NoData encoding, dynamic range, and cross-check the reconstructed 100 m values against NB04B sampled targets for each mosaic. The code automatically tests several plausible uint16 denominators but **refuses to accept a scale** unless the agreement is good. Do not retrain anything if this fails.

Generate site/mosaic **side-by-side Planet RGB, original UAV orthomosaic RGB, and continuous UAV probability** panels with one common geographic extent at native pixels. The images may have different sensor dates and radiometry. The purpose is to check obvious geometric shifts, not to infer identity from colours.

*Important*: UAV 2020 and Planet 2021 for Yoko; one-year mismatch is logged. A visually plausible image is not proof of subpixel registration; if the overlay looks shifted, fix georeferencing upstream before extracting labels.


In [ ]:
# 06 — Authoritative UAV teacher paths and raster metadata
import numpy as np,pandas as pd,rasterio,matplotlib.pyplot as plt, json,math
from rasterio.warp import reproject,Resampling,transform_bounds
from rasterio.windows import from_bounds,Window
from rasterio.features import rasterize
from affine import Affine
from shapely.geometry import box
from IPython.display import display

UAV_TEACHERS=FINAL/'Outputs/V3_1/03_M2_warmstart_c6a33504c37efb2ebb84_full_mosaic/probability_maps'
UAV_MOSAICS={
 'Baego':['Baego.tif'],
 'MOS':['Mos_1_2_transparent_mosaic_group1_georef.tif','Mos_5_6_transparent_mosaic_group1_georef.tif'],
 'Yangambi':['Yangambi_georef.tif'],
 'Yoko':['Yoko_kvo_georef.tif']
}
MANIFEST_FILE=FINAL/'Outputs/V3_1/00_input_audit/00_input_manifest.json'
assert MANIFEST_FILE.exists(),MANIFEST_FILE
uav_manifest=json.loads(MANIFEST_FILE.read_text())
MOSAIC_PATH={x['file']:Path(x['path']) for x in uav_manifest['mosaics']}
for site,names in UAV_MOSAICS.items():
    for name in names:
        assert name in MOSAIC_PATH and MOSAIC_PATH[name].is_file(), ('Missing orthomosaic',site,name)
def teacher_file(name):
    return UAV_TEACHERS/f'{Path(name).stem}_M2_warmstart_epoch02_probability_u16_FULL.tif'
SOURCE_TABLE=[]
for site,names in UAV_MOSAICS.items():
    for name in names:
        path=teacher_file(name)
        assert path.is_file(),f'Authoritative UAV model output missing: {path}'
        with rasterio.open(path) as ds:
            assert ds.count>=1 and ds.crs and ds.width>0 and ds.height>0
            # Distributed window samples for a rapid encoding diagnostic.
            samples=[]
            for frac_y,frac_x in ((.25,.25),(.5,.5),(.75,.75),(.3,.7),(.7,.3)):
                win=Window(max(0,int(ds.width*frac_x)-64),max(0,int(ds.height*frac_y)-64),
                           min(128,ds.width),min(128,ds.height))
                samples.append(ds.read(1,window=win))
            z=np.concatenate([a.reshape(-1) for a in samples])
            SOURCE_TABLE.append(dict(site=site,mosaic=name,crs=str(ds.crs),
                width=ds.width,height=ds.height,res_x=ds.res[0],res_y=ds.res[1],
                dtype=ds.dtypes[0],nodata=ds.nodata,
                sample_min=float(np.min(z)),sample_p99=float(np.percentile(z,99)),
                sample_max=float(np.max(z)),file=str(path)))
display(pd.DataFrame(SOURCE_TABLE).drop(columns='file'))
pd.DataFrame(SOURCE_TABLE).to_csv(OUT/'06_uav_teacher_audit.csv',index=False)
print('Five original M2 warmstart probability maps and orthomosaics located. No labels modified.')


## 07 · RAM-safe validation against NB04B

**The previous two implementations could trigger large memory use when resampling multi-gigapixel UAV probability rasters, even with GDAL VRT.** We now avoid original full-resolution rasters entirely. NB04 already produced **10, 30, 50, 75, 100 and 125 m cached UAV aggregates** on exactly the same satellite lattice. The cache stores continuous **mean Musanga probability (band 1)**, thresholded cover (band 2), and valid UAV coverage (band 3), all normalized to [0,1].

This section reads only the cached **100 m** files and compares them with known NB04B sample fractions; the existing **50 m** cache will be used downstream. No uint16 scale guessing or expensive native-raster reprojection. Rerun Sections 00–02, 06 and 07 after reconnecting to the new notebook.


In [ ]:
# 07 — FAST: reuse verified NB04 cached 10/50/100 m UAV aggregates.
# NO source UAV raster opened. NO WarpedVRT. NO full-resolution resampling.
from pathlib import Path
import rasterio,numpy as np,pandas as pd,math
from affine import Affine
from IPython.display import display

AGG_DIR=FINAL/'Outputs/V3_1/04_satellite_full_warmstart/cache/aggregates'
assert AGG_DIR.is_dir(),AGG_DIR
REFERENCE_DIAGNOSTICS=[]
UAV_SCALE={name:1.0 for names in UAV_MOSAICS.values() for name in names}

def satellite_grid(site,scale):
    path=NB04/'RAW_ANNUAL'/f'{site}_RAW_ANNUAL_{EMBEDDING_YEAR[site]}_10m.tif'
    with rasterio.open(path) as ds:
        assert ds.crs and abs(ds.res[0]-10)<.05
        return dict(crs=ds.crs,transform=ds.transform*Affine.scale(scale/10,scale/10),
                    width=int(math.ceil(ds.width*10/scale)),
                    height=int(math.ceil(ds.height*10/scale)))

def agg_file(name,scale):
    return AGG_DIR/f'{Path(name).stem}_{scale}m.tif'

def read_cached_teacher(name,site,scale):
    """Existing NB04 cache: band 1 mean probability, 2 cover>0.5, 3 UAV coverage.
    All three are ALREADY in [0,1], aligned to the canonical satellite grid.
    """
    path=agg_file(name,scale)
    assert path.is_file(),f'Cached {scale}m UAV aggregate missing: {path}'
    g=satellite_grid(site,scale)
    with rasterio.open(path) as src:
        assert src.count>=3,(name,scale,'unexpected bands',src.count)
        assert src.crs==g['crs'],(name,scale,'CRS mismatch')
        assert src.width==g['width'] and src.height==g['height'],(name,scale,'grid dimensions mismatch')
        assert src.transform.almost_equals(g['transform'],precision=1e-6),(name,scale,'grid alignment mismatch')
        a=src.read([1,3],out_dtype='float32')
    values=a[0];cov=a[1]
    return values,cov,g

for site,names in UAV_MOSAICS.items():
    for name in names:
        ref=DF.loc[(DF.site==site)&(DF.mosaic==name)]
        assert len(ref)>0,(site,name,'missing NB04B examples')
        values,cov,g=read_cached_teacher(name,site,100)
        rr=ref.row.to_numpy(dtype=int)
        cc=ref.col.to_numpy(dtype=int)
        valid_grid=(rr>=0)&(cc>=0)&(rr<g['height'])&(cc<g['width'])
        rr,cc=rr[valid_grid],cc[valid_grid]
        known=ref.mean_probability.to_numpy(dtype=float)[valid_grid]
        pred=values[rr,cc];coverage=cov[rr,cc]
        valid=(coverage>=0.90)&np.isfinite(pred)&np.isfinite(known)
        assert valid.sum()>=8,(name,'too few valid cells',int(valid.sum()))
        mae=float(np.mean(np.abs(pred[valid]-known[valid])))
        max_diff=float(np.max(np.abs(pred[valid]-known[valid])))
        REFERENCE_DIAGNOSTICS.append(dict(site=site,mosaic=name,n=int(valid.sum()),
            source='NB04_cached_100m',MAE_fraction=mae,max_abs_difference=max_diff,
            median_reference=float(np.median(known[valid])),
            median_cache=float(np.median(pred[valid]))))
        print('VERIFIED',site,Path(name).stem,'n',int(valid.sum()),'MAE',round(mae,7),
              'maximum',round(max_diff,7),flush=True)
        assert mae<0.001,(name,'NB04B cache mismatch',mae)
        del values,cov
CALIBRATION=pd.DataFrame(REFERENCE_DIAGNOSTICS)
display(CALIBRATION.round(7))
CALIBRATION.to_csv(OUT/'07_teacher_encoding_validation.csv',index=False)
print('Safe: all comparisons use small cached 100m rasters. No native UAV probability raster was read.')
print('Section 09 will use existing cached 50m probabilities; do not resample large UAV source files.')


## 08 · Low-memory visual alignment and teacher-distribution QA

The original teacher panels appeared **nearly uniformly pale yellow**, which is the `magma` colour for probability approximately 1. The previous preview did not use the aggregate raster's valid-UAV coverage band and could therefore display invalid/background values as real predictions.

This revised window-only preview reads the **10 m cached mean probability (band 1)** **and UAV coverage (band 3)** together. It masks coverage below 90%, shades invalid pixels gray, and prints within-window p05/p50/p95 and percentages near 0 or 1. It also prints the selected 100 m reference fraction for context. **Do not interpret or train on 10 m probability values if the diagnostics still show saturation at 1.** This remains display-only QA, not a full source-raster calibration.


In [ ]:
# 08 — Bounded window previews ONLY. No source-wide reproject/read.
from rasterio.windows import from_bounds as bounds_window
from rasterio.warp import transform_bounds
from rasterio.enums import Resampling

PANEL_M=350
PANEL_N=420

def preview_window(filepath,world_bounds,crs,bands,resampling=Resampling.bilinear):
    """Read one bounding-box window, decimated straight into a small array."""
    with rasterio.open(filepath) as ds:
        assert ds.crs and max(bands)<=ds.count,(filepath,bands)
        if ds.crs==crs:
            bb=world_bounds
        else:
            bb=transform_bounds(crs,ds.crs,*world_bounds,densify_pts=21)
        win=bounds_window(*bb,transform=ds.transform)
        # Protect against invalid geometry/projection or accidental huge windows.
        if win.width<=0 or win.height<=0:
            raise ValueError(f'Empty window: {filepath}')
        if win.width>1e7 or win.height>1e7:
            raise ValueError(f'Unreasonable source window: {filepath}, {win}')
        data=ds.read(bands,window=win,out_shape=(len(bands),PANEL_N,PANEL_N),
                     boundless=True,fill_value=0,resampling=resampling,
                     out_dtype='float32')
    return data

def show_rgb(cube):
    valid=np.isfinite(cube).all(axis=0)&(cube>0).all(axis=0)
    img=np.ones((*valid.shape,3),dtype='float32')
    if valid.sum()>5:
        lo=np.percentile(cube[:,valid],2,axis=1)
        hi=np.percentile(cube[:,valid],98,axis=1)
        stretch=(cube-lo[:,None,None])/(hi[:,None,None]-lo[:,None,None]+1e-6)
        img[valid]=np.clip(stretch.transpose(1,2,0)[valid],0,1)
    return img

fig,axes=plt.subplots(5,3,figsize=(13,20),layout='constrained')
prob_diagnostics=[]
idx=0
for site,names in UAV_MOSAICS.items():
    for name in names:
        ref=DF.loc[(DF.site==site)&(DF.mosaic==name)].sort_values('mean_probability')
        assert len(ref),name
        pick=ref.iloc[len(ref)//2]
        g=satellite_grid(site,100)
        cx,cy=g['transform']*(float(pick.col)+.5,float(pick.row)+.5)
        half=PANEL_M/2
        bounds=(cx-half,cy-half,cx+half,cy+half)
        # Native UAV RGB is window-read, never loaded as a complete mosaic.
        uav_rgb=preview_window(MOSAIC_PATH[name],bounds,g['crs'],[1,2,3])
        # Planet raw median RGB: R/G/B are bands 15,14,13.
        planet_rgb=preview_window(planet_file(site,'Annual'),bounds,g['crs'],[15,14,13])
        # Existing 10m NB04 UAV aggregate; band 1 is normalized mean probability.
        # NB04 aggregate bands: 1 mean probability, 2 fraction >0.5, 3 valid UAV coverage.
        # Read mask and target together: otherwise invalid/background pixels may
        # be rendered as high probability, creating a deceptively solid yellow map.
        uc=preview_window(agg_file(name,10),bounds,g['crs'],[1,3],
                          resampling=Resampling.nearest)
        prob,coverage=uc[0],uc[1]
        good=np.isfinite(prob)&np.isfinite(coverage)&(coverage>=0.90)&(prob>=0)&(prob<=1)
        if good.any():
            pv=prob[good]
            stats=dict(site=site,mosaic=name,n_valid=int(good.sum()),
                       window_valid_pct=100*float(good.mean()),
                       p05=float(np.percentile(pv,5)),p50=float(np.median(pv)),
                       p95=float(np.percentile(pv,95)),share_gt_09_pct=100*float(np.mean(pv>=.9)),
                       share_lt_01_pct=100*float(np.mean(pv<=.1)),
                       reference_100m_fraction=float(pick.mean_probability))
        else:
            stats=dict(site=site,mosaic=name,n_valid=0,window_valid_pct=0,
                       p05=np.nan,p50=np.nan,p95=np.nan,share_gt_09_pct=np.nan,
                       share_lt_01_pct=np.nan,reference_100m_fraction=float(pick.mean_probability))
        prob_diagnostics.append(stats)
        axes[idx,0].imshow(show_rgb(uav_rgb))
        axes[idx,1].imshow(show_rgb(planet_rgb))
        cmap=plt.get_cmap('magma').copy()
        cmap.set_bad('#dddddd')
        axes[idx,2].imshow(np.ma.masked_where(~good,prob),cmap=cmap,
                           vmin=0,vmax=1,interpolation='nearest')
        axes[idx,0].set_title(f'{site} {Path(name).stem}: UAV RGB')
        axes[idx,1].set_title(f'Planet {PLANET_YEAR[site]} raw median RGB')
        axes[idx,2].set_title('UAV fraction (10m, valid UAV pixels only)')
        for ax in axes[idx]: ax.axis('off')
        idx+=1
        print('PREVIEW',site,Path(name).stem,'finished',flush=True)
DIAG08=pd.DataFrame(prob_diagnostics)
display(DIAG08.round(3))
DIAG08.to_csv(OUT/'08_teacher_probability_diagnostics.csv',index=False)
print('WARNING if a 10m map has >90% of valid pixels near 1: possible masking,')
print('scaling or band-semantic issue; investigate before training a pixel U-Net.')
fig.savefig(FIG/'08_uav_planet_teacher_alignment.png',dpi=135,bbox_inches='tight')
plt.show()
plt.close(fig)
print('Window-only QA complete: examine broad landscape alignment, not individual crowns.')


## 09 · Build the 50 m training-target table from **existing verified NB04 cached aggregates**

Avoid rerunning expensive UAV raster aggregation. NB04 already stored canonical 50 m continuous Musanga fractions and valid-pixel fractions for each original mosaic. Section 07 checks the 100 m caches against NB04B samples; this section reads the corresponding 50 m caches, intersects them with `ROI.shp`, and keeps cells with at least 90% valid UAV coverage and 90% ROI membership. Source cache is not modified. This is independent of Planet availability; Section 10 checks its patches.


In [ ]:
# 09 — Build continuous 50m UAV labels (per-mosaic), enforce NB04B-style ROI/coverage rules.
import geopandas as gpd
from rasterio.features import rasterize
ROI_SOURCE=Path(uav_manifest['roi'])
assert ROI_SOURCE.is_file()
ROI_GDF=gpd.read_file(ROI_SOURCE)
assert ROI_GDF.crs and not ROI_GDF.empty
TARGET_SCALE=50
MIN_UAV_COVERAGE=.90
MIN_ROI_FRAC=.90
SUPERSAMPLE=4
TARGETS=[]
for site,names in UAV_MOSAICS.items():
    g=satellite_grid(site,TARGET_SCALE)
    polys=ROI_GDF.to_crs(g['crs'])
    fine_transform=g['transform']*Affine.scale(1/SUPERSAMPLE,1/SUPERSAMPLE)
    fine=rasterize(((geom,1) for geom in polys.geometry if geom is not None and not geom.is_empty),
        out_shape=(g['height']*SUPERSAMPLE,g['width']*SUPERSAMPLE),
        transform=fine_transform,fill=0,dtype='uint8')
    roif=fine.reshape(g['height'],SUPERSAMPLE,g['width'],SUPERSAMPLE).mean(axis=(1,3))
    del fine
    for name in names:
        probs,coverage,_=read_cached_teacher(name,site,TARGET_SCALE)
        ok=(coverage>=MIN_UAV_COVERAGE)&(roif>=MIN_ROI_FRAC)&np.isfinite(probs)&(probs>=0)&(probs<=1)
        yy,xx=np.where(ok)
        for row,col in zip(yy,xx):
            x,y=g['transform']*(int(col)+.5,int(row)+.5)
            TARGETS.append(dict(site=site,mosaic=name,group=Path(name).stem,
                year_uav=EMBEDDING_YEAR[site],year_planet=PLANET_YEAR[site],
                scale_m=TARGET_SCALE,row=int(row),col=int(col),
                x=float(x),y=float(y),crs=str(g['crs']),
                mean_probability=float(probs[row,col]),
                uav_coverage=float(coverage[row,col]),
                roi_fraction=float(roif[row,col])))
        print('50m supervised cells',site,name,len(yy),flush=True)
assert TARGETS, 'No 50m cells: inspect calibration / ROI / CRS'
T50=pd.DataFrame(TARGETS)
assert not T50.duplicated(['site','mosaic','row','col']).any()
T50.to_parquet(OUT/'09_uav_targets_50m.parquet',index=False)
display(T50.groupby(['site','group']).agg(cells=('mean_probability','size'),
    mean_fraction=('mean_probability','mean'),
    max_fraction=('mean_probability','max'),
    min_uav_coverage=('uav_coverage','min')).round(3))
print('Saved verified 50m UAV-derived labels:',OUT/'09_uav_targets_50m.parquet')


## 10 · Full 50 m spatial-feature extraction — restartable and RAM-aware

Runs on **all 14,454 target cells** from Section 09 (not just 300 per site). Reads **native 4.77 m** Planet reflectance and NIR texture neighborhoods and **10 m** embedding neighborhoods without creating coarsened satellite rasters. The target remains the existing continuous UAV-derived Musanga fraction on the 50 m grid.

**Features**: Planet B/G/R/N patch means + SD (21×21 Planet pixels, ~100 m), medians of four existing Planet NIR texture layers, and **all 64** embedding channel values at the patch centre, context means and SDs (11×11 embedding pixels, ~110 m). The wider neighborhood supplies spatial context; it does **not** change the 50 m response support.

**Restart behaviour**: deterministic 200-cell chunks, written as Parquet with an entry for **every** input cell, including rejected patches and reason. Completed chunks are checked and skipped on rerun. Each run uses an input/configuration fingerprint (target coordinates, labels, source file metadata and window settings); changes create a new cache namespace, preventing stale reuse. A consolidated feature table includes only accepted rows, and a companion QA file records rejected cells. Source imagery and previous 300-cell pilot are preserved.

**Memory/I/O**: only small raster windows are read. Opens each site's three sources once; no full mosaics and no 64-channel cubes across the whole forest. Outputs progress after each chunk. The larger MOS dataset is extracted incrementally.

You can safely interrupt after any saved batch. Reopen Colab and rerun initialization, Section 09 (or load its existing Parquet file), then Section 10. No CNN/model training occurs here.


In [ ]:
# 10 — Full, restartable native Planet + embedding spatial feature extraction.
# Requires setup cells (FINAL, OUT, NB04...), Section 01 (planet_file/embed_file),
# and Section 09 (T50) or the existing Section 09 Parquet target file.
from pathlib import Path
import hashlib,json,os,gc,time
import numpy as np,pandas as pd,rasterio
from rasterio.windows import Window
from pyproj import Transformer
from IPython.display import display

TARGET_FILE=OUT/'09_uav_targets_50m.parquet'
if 'T50' not in globals():
    assert TARGET_FILE.is_file(),f'Run Section 09 first: {TARGET_FILE}'
    T50=pd.read_parquet(TARGET_FILE)
assert len(T50)>0 and {'site','mosaic','row','col','x','y','crs','mean_probability'}.issubset(T50.columns)
BATCH_CELLS=200
PLANET_PATCH=21  # 21 x 4.77m ~100m, centred on 50m response cell
EMB_PATCH=11     # 11 x 10m =110m embedding context
MIN_PATCH_VALID=.80
MAX_NEW_BATCHES=None  # Set e.g. 3 for a short smoke test; None = all outstanding.
FORCE_NEW_EXTRACTION=False  # True adds a new cache version; normally leave False.
EXTRACTION_VERSION='v2_native_planet_embed_20261024'
COLS_KEY=['site','mosaic','row','col']
T50=T50.sort_values(COLS_KEY,kind='stable').reset_index(drop=True)
assert not T50.duplicated(COLS_KEY).any(),'Duplicated input target cell keys'

sources={}
for site in SITES:
    sources[site]=dict(annual=planet_file(site,'Annual'),
                       texture=planet_file(site,'Texture'),
                       embeddings=embed_file(site))
    for path in sources[site].values():assert path.is_file(),path

# Source identity (path + size + modification nanoseconds) and every target
# value/position must match for a chunk to be reusable.
config=dict(version=EXTRACTION_VERSION,planet_patch=PLANET_PATCH,
            embedding_patch=EMB_PATCH,min_patch_valid=MIN_PATCH_VALID,
            chunk_size=BATCH_CELLS,
            source_identity={s:{k:dict(path=str(v),size=v.stat().st_size,
                mtime_ns=v.stat().st_mtime_ns) for k,v in paths.items()}
                for s,paths in sources.items()},
            targets_sha256=hashlib.sha256(
                pd.util.hash_pandas_object(
                    T50[['site','mosaic','row','col','x','y','crs','mean_probability']],
                    index=False).values.tobytes()).hexdigest())
config_json=json.dumps(config,sort_keys=True)
run_id=hashlib.sha256(config_json.encode()).hexdigest()[:16]
if FORCE_NEW_EXTRACTION:
    # Manual new run without changing source files
    EXTRACTION_VERSION+='__manual'
    config['version']=EXTRACTION_VERSION
    run_id=hashlib.sha256(json.dumps(config,sort_keys=True).encode()).hexdigest()[:16]
CACHE10=OUT/'10_full_spatial_features_chunks'/run_id
CACHE10.mkdir(parents=True,exist_ok=True)
manifest_path=CACHE10/'config.json'
if manifest_path.exists():
    assert json.loads(manifest_path.read_text())==config,'Cached run settings changed unexpectedly'
else:
    manifest_path.write_text(json.dumps(config,indent=2,sort_keys=True))

def one_patch(ds,bands,size,x,y,source_crs,transformer):
    tx,ty=transformer.transform(x,y)
    px,py=(~ds.transform)*(tx,ty)
    # Use a deterministic centre aligned to nearest native pixel.
    c=int(np.floor(px));r=int(np.floor(py))
    h=size//2
    return ds.read(bands,window=Window(c-h,r-h,size,size),
                   boundless=True,fill_value=0,out_dtype='float32')

def extract_batch(site,block):
    out=[]
    with rasterio.open(sources[site]['annual']) as pds,\
         rasterio.open(sources[site]['texture']) as tds,\
         rasterio.open(sources[site]['embeddings']) as eds:
        trans={id(ds):Transformer.from_crs(block.crs.iloc[0],ds.crs,always_xy=True)
               for ds in (pds,tds,eds)}
        for rec in block.itertuples(index=False):
            base=dict(site=site,mosaic=rec.mosaic,group=rec.group,
                      row=int(rec.row),col=int(rec.col),
                      target_fraction=float(rec.mean_probability),
                      x=float(rec.x),y=float(rec.y),crs=str(rec.crs))
            try:
                if str(rec.crs)!=str(block.crs.iloc[0]):
                    raise ValueError('MIXED_CRS_IN_BATCH')
                p=one_patch(pds,[13,14,15,16],PLANET_PATCH,rec.x,rec.y,
                            rec.crs,trans[id(pds)])
                t=one_patch(tds,list(range(1,10)),PLANET_PATCH,rec.x,rec.y,
                            rec.crs,trans[id(tds)])
                e=one_patch(eds,list(range(1,65)),EMB_PATCH,rec.x,rec.y,
                            rec.crs,trans[id(eds)])
                pm=np.isfinite(p).all(axis=0)&(p>0).all(axis=0)
                em=np.isfinite(e).all(axis=0)&np.any(e!=0,axis=0)
                tv=np.isfinite(t).all(axis=0)&(t>=0).all(axis=0)
                base.update(planet_valid=float(pm.mean()),
                            embedding_valid=float(em.mean()),
                            planet_texture_valid=float(tv.mean()))
                if pm.mean()<MIN_PATCH_VALID:
                    base.update(accepted=False,reason='PLANET_LOW_COVERAGE')
                elif em.mean()<MIN_PATCH_VALID:
                    base.update(accepted=False,reason='EMBEDDING_LOW_COVERAGE')
                elif tv.mean()<MIN_PATCH_VALID:
                    base.update(accepted=False,reason='TEXTURE_LOW_COVERAGE')
                else:
                    px=p[:,pm]
                    ex=e[:,em]
                    center=e[:,EMB_PATCH//2,EMB_PATCH//2]
                    if not np.isfinite(center).all():
                        base.update(accepted=False,reason='EMBEDDING_CENTRE_INVALID')
                    else:
                        base.update(accepted=True,reason='OK')
                        for i,band in enumerate(['B','G','R','N']):
                            base[f'PLANET_mean_{band}']=float(px[i].mean())
                            base[f'PLANET_std_{band}']=float(px[i].std())
                        for i,band in enumerate(['NIR_CV_r1','NIR_CV_r3',
                                                  'NIR_CV_r7','NIR_highpass']):
                            v=t[i,tv]
                            base[f'PLANET_texture_{band}']=float(np.median(v))
                        means=ex.mean(axis=1);stds=ex.std(axis=1)
                        for j in range(64):
                            base[f'EMB_center_{j+1:02d}']=float(center[j])
                            base[f'EMB_mean_{j+1:02d}']=float(means[j])
                            base[f'EMB_sd_{j+1:02d}']=float(stds[j])
            except (OSError,ValueError,rasterio.errors.RasterioIOError) as exc:
                base.update(accepted=False,reason='READ_ERROR',error=str(exc)[:250])
            out.append(base)
    return pd.DataFrame(out)

# Separate every accepted/rejected target record; one output row per input cell.
# On a crash, only the currently active unsaved chunk needs recomputing.
work=[]
for site in SITES:
    site_cells=T50.loc[T50.site==site].reset_index(drop=True)
    for offset in range(0,len(site_cells),BATCH_CELLS):
        block=site_cells.iloc[offset:offset+BATCH_CELLS].copy()
        key=f'{site}_{offset:06d}_{len(block):03d}'
        work.append((site,block,CACHE10/f'{key}.parquet'))
print('FULL DATASET:',len(T50),'cells;',len(work),'restartable chunks;',
      'cache',CACHE10,flush=True)
print('Existing 300/site pilot is untouched.',flush=True)

new_batches=0
start=time.monotonic()
for k,(site,block,path) in enumerate(work,1):
    if path.exists():
        try:
            old=pd.read_parquet(path,columns=COLS_KEY+['accepted','reason'])
            expect=block[COLS_KEY].reset_index(drop=True)
            if len(old)==len(block) and old[COLS_KEY].reset_index(drop=True).equals(expect):
                print(f'[{k}/{len(work)}] CACHED {site} {path.name}: '
                      f'{int(old.accepted.sum())}/{len(old)} accepted',flush=True)
                continue
        except Exception as exc:
            print('INCOMPLETE CACHE ignored:',path.name,repr(exc),flush=True)
        # Do not trust a partial or stale cache; regenerate this block.
    if MAX_NEW_BATCHES is not None and new_batches>=MAX_NEW_BATCHES:
        print('Reached MAX_NEW_BATCHES; rerun to continue.',flush=True)
        break
    t0=time.monotonic()
    data=extract_batch(site,block)
    assert len(data)==len(block) and data[COLS_KEY].reset_index(drop=True).equals(
        block[COLS_KEY].reset_index(drop=True)),('Unexpected extraction row mismatch',site,k)
    tmp=path.with_suffix('.parquet.partial')
    data.to_parquet(tmp,index=False)
    os.replace(tmp,path)
    new_batches+=1
    print(f'[{k}/{len(work)}] SAVED {site} {path.name}: '
          f'{int(data.accepted.sum())}/{len(data)} accepted '
          f'({time.monotonic()-t0:.1f}s; overall {time.monotonic()-start:.1f}s)',
          flush=True)
    del data
    gc.collect()

# Only consolidate if every chunk is present (otherwise keep reusable batches).
missing=[str(path) for _,_,path in work if not path.exists()]
if missing:
    print('EXTRACTION PARTIAL:',len(missing),'remaining chunks. Rerun Section 10 to resume.')
else:
    # Bounded accumulated table: 14,454 rows × ~220 columns, modest in memory.
    all_rows=pd.concat([pd.read_parquet(path) for _,_,path in work],ignore_index=True)
    assert len(all_rows)==len(T50)
    assert all_rows[COLS_KEY].equals(T50[COLS_KEY]),'Joined rows out of order'
    accepted=all_rows.loc[all_rows.accepted].copy()
    rejected=all_rows.loc[~all_rows.accepted].copy()
    full_file=OUT/'10_full_spatial_features_50m.parquet'
    qa_file=OUT/'10_full_spatial_features_rejections_50m.parquet'
    report=OUT/'10_full_spatial_features_site_report.csv'
    accepted.to_parquet(full_file,index=False)
    rejected.to_parquet(qa_file,index=False)
    report_df=all_rows.groupby('site').agg(
        total=('accepted','size'),accepted=('accepted','sum'),
        mean_planet_valid=('planet_valid','mean'),
        mean_embedding_valid=('embedding_valid','mean')).reset_index()
    report_df['rejected']=report_df.total-report_df.accepted
    display(report_df.round(3))
    print('Rejection reasons:')
    display(all_rows.groupby(['site','reason']).size().rename('cells').reset_index())
    report_df.to_csv(report,index=False)
    print('COMPLETE:',len(accepted),'accepted /',len(all_rows),'targets.')
    print('Features:',full_file)
    print('QA rejected:',qa_file)
    print('Summary:',report)
    print('No model fit; preserve geographic groups for spatial holdout.')
